# RAG-Anything — Ingestion sur Google Colab (GPU)

Ce notebook ingère les documents PDF dans le knowledge graph LightRAG **sur GPU Colab**,
puis télécharge le storage généré pour l'utiliser en local.

**Stack :**
- LLM extraction & réponses : `llama3.1:8b` — compatible T4 16GB (~5GB VRAM)
- Vision : `llava:7b`
- Embeddings : `nomic-embed-text`

**Améliorations v4 :**
- Chunks de **1200 tokens** (vs 400) → les paragraphes restent complets, meilleure qualité de réponses
- API native Ollama `/api/chat` avec `num_ctx=8192` → contexte complet transmis au LLM
- Instance RAGAnything en **singleton** → pas de rechargement à chaque requête

**Prérequis :** Activer le GPU dans `Exécution > Modifier le type d'exécution > GPU`

In [ ]:
# ── Cellule 1 : Installer Ollama ────────────────────────────────────────────
!apt-get install -y zstd -q
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# ── Cellule 2 : Démarrer Ollama + télécharger les modèles ───────────────────
import subprocess, time

# Démarrer le serveur Ollama en arrière-plan
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print("✅ Ollama démarré")

# llama3.1:8b (~5GB VRAM) — compatible T4 16GB
for model in ["llama3.1:8b", "llava:7b", "nomic-embed-text"]:
    print(f"\nPull {model}...")
    !ollama pull {model}

print("\n✅ Tous les modèles sont prêts")


In [ ]:
# ── Cellule 3 : Cloner le projet ────────────────────────────────────────────
%cd /content
!rm -rf rag-anything-project
!git clone https://github.com/Abdelmajid-BARANI/rag-anything-project.git
%cd rag-anything-project
!pip install -r requirements.txt -q
print("✅ Projet installé")

In [ ]:
# ── Cellule 4 : Uploader les PDFs ───────────────────────────────────────────
import os
from google.colab import files

data_path = "/content/rag-anything-project/donnees_rag"
os.makedirs(data_path, exist_ok=True)

print("Sélectionner les fichiers PDF à ingérer :")
uploaded = files.upload()

for fname, data in uploaded.items():
    dest = os.path.join(data_path, fname)
    with open(dest, "wb") as f:
        f.write(data)

pdfs = os.listdir(data_path)
print(f"\n✅ {len(pdfs)} fichier(s) uploadé(s) dans donnees_rag/ :")
for p in pdfs:
    print(f"  - {p}")

In [ ]:
# ── Cellule 5 : Lancer l'ingestion (GPU CUDA) ────────────────────────────────
import os, shutil

# Supprimer l'ancien index (chunk_size a changé : 400 → 1200 tokens)
storage_path = '/content/rag-anything-project/data/rag_anything_storage'
if os.path.exists(storage_path):
    shutil.rmtree(storage_path)
    print('🗑️  Ancien index supprimé')

# llama3.1:8b — compatible T4 16GB (~5GB VRAM)
os.environ['PARSE_METHOD']         = 'auto'          # parsing multimodal (texte + tableaux)
os.environ['MINERU_DEVICE']        = 'cuda'          # GPU pour MinerU
os.environ['OLLAMA_LLM_MODEL']     = 'llama3.1:8b'
os.environ['OLLAMA_EXTRACT_MODEL'] = 'llama3.1:8b'
os.environ['OLLAMA_VISION_MODEL']  = 'llava:7b'
os.environ['OLLAMA_EMBED_MODEL']   = 'nomic-embed-text'

# chunk_token_size=1200 est configuré dans src/ingestion/rag_anything_pipeline.py

%cd /content/rag-anything-project
!python src/ingestion/rag_anything_pipeline.py


In [ ]:
# ── Cellule 6 : Vérifier le résultat de l'ingestion ─────────────────────────
import json, os

storage = "/content/rag-anything-project/data/rag_anything_storage"

def load_json(path):
    if not os.path.exists(path):
        print(f"⚠️  Fichier absent : {os.path.basename(path)}")
        return {}
    with open(path) as f:
        return json.load(f)

docs      = load_json(f"{storage}/kv_store_doc_status.json")
chunks    = load_json(f"{storage}/kv_store_text_chunks.json")
entities  = load_json(f"{storage}/kv_store_full_entities.json")
relations = load_json(f"{storage}/kv_store_full_relations.json")

print(f"Documents ingérés  : {len(docs)}")
print(f"Chunks de texte    : {len(chunks)}")
print(f"Entités extraites  : {len(entities)}")
print(f"Relations extraites: {len(relations)}")

if not docs:
    print("\n❌ Aucun document ingéré — relancer la cellule 5")
elif len(entities) <= len(docs):
    print("\n⚠️  Extraction d'entités insuffisante — vérifier les logs Ollama")
else:
    print("\n✅ Knowledge graph peuplé correctement")


In [ ]:
# ── Cellule 7 : Diagnostic Ollama (exécuter si ingestion incomplète) ─────────
import requests, os

# 1. Modèles chargés dans Ollama
print("=== Modèles Ollama disponibles ===")
!ollama list

# 2. Test de connectivité
print("\n=== Test connexion Ollama ===")
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"✅ Ollama répond — {len(models)} modèle(s) : {models}")
except Exception as e:
    print(f"❌ Ollama inaccessible : {e}")

# 3. GPU disponible et mémoire
print("\n=== GPU disponible ===")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader 2>/dev/null || echo "Pas de GPU detecte"

# 4. Dernières lignes du log pipeline
print("\n=== Dernières lignes du log RAG ===")
log_path = "/content/rag-anything-project/logs/rag_anything.log"
if os.path.exists(log_path):
    !tail -40 {log_path}
else:
    print("Log absent — la pipeline ne s'est peut-etre pas lancee correctement")


In [ ]:
# ── Cellule 8 : Télécharger le knowledge graph ──────────────────────────────
import shutil
from google.colab import files

shutil.make_archive(
    "/content/rag_storage_backup",
    "zip",
    "/content/rag-anything-project/data/rag_anything_storage"
)

print("Téléchargement du storage...")
files.download("/content/rag_storage_backup.zip")
print("\n✅ Dézipper dans data/rag_anything_storage/ en local pour utiliser le RAG")